# Summary Analysis and Figure Generation

This notebook generates figures for the manuscript from the processed single-cell RNA-seq data of the discovery cohort (initial cohort) and validates the SOX9+ basal-like gene signature in the validation cohort. It covers cell type composition analysis, epithelial subtype characterization, and cross-cohort gene signature validation.

**Reproducibility note**: Cell type annotations and UMAP coordinates used in this notebook are loaded from pre-computed `.h5ad` files produced by `01_clustering.ipynb`, and are not recalculated here. Gene signature scores (`sc.tl.score_genes`) involve random sampling of a reference gene set; exact score values may vary slightly across runs, but the relative ordering and statistical conclusions are robust to this variation.

In [1]:
import importlib.metadata
import sys

packages = [
    'scanpy', 'anndata', 'pandas', 'numpy',
    'matplotlib', 'seaborn', 'scipy', 'scikit-learn',
]

print(f'Python: {sys.version}')
for pkg in packages:
    try:
        print(f'{pkg}: {importlib.metadata.version(pkg)}')
    except importlib.metadata.PackageNotFoundError:
        print(f'{pkg}: not found')

Python: 3.9.18 | packaged by conda-forge | (main, Dec 23 2023, 16:35:41) 
[Clang 16.0.6 ]
scanpy: 1.10.1
anndata: 0.10.8
pandas: 2.2.2
numpy: 1.26.4
matplotlib: 3.9.1
seaborn: 0.13.2
scipy: 1.13.1
scikit-learn: 1.1.3


## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import anndata
import scanpy as sc
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rcParams['font.family'] = 'Times'

from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from matplotlib.pyplot import get_cmap

from scipy.stats import mannwhitneyu
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, f1_score
from sklearn.metrics import roc_curve

from datetime import datetime

os.makedirs('figures', exist_ok=True)
d = datetime.now().strftime('%y%m%d')
print(d)

## 2. Cell Type Composition Analysis

Load the final integrated AnnData object from the discovery cohort and generate figures for global cell type composition: UMAP, dotplot of marker genes, and stacked bar charts of cell type proportions per sample.

In [ ]:
# Each section reloads adata independently from the source h5ad to avoid
# unintended side effects from in-place modifications in preceding sections.
adata = anndata.read_h5ad('./11_clustering/final.h5ad')

In [ ]:
adata.obs.shape

In [ ]:
pd.DataFrame(adata.obs['cell_type'].unique())

In [ ]:
temp = adata[adata.obs['cell_type'] != 'Doublet'].copy()

gene_vec = [
'PTPRC', # IMMUNE
'CD3D', 'CD3E', # T
'GNLY', # NK
'CD79A', 'MS4A1', # B
'JCHAIN', # PLASMA
'LYZ', 'CD68', # MACROPHAGE
'TPSAB1', 'TPSB2', # MAST
'PECAM1', 'VWF', # ENDOTHELIAL
'RGS5', 'PDGFRB', # PERICYTE
'ACTA2', 'MYH11', # VSMC
'DCN', 'PDGFRA', # FIBROBLAST
'KRT8', 'KRT18', # EPITHELIAL
'FOXJ1', 'PIFO' # CILIATED
]

celltype_order = [
    'T/NK',
    'Macrophage',
    'Mast',
    'Endothelial',
    'SMC/Pericyte',
    'Fibroblast',
    'Epithelial',
    'Ciliated',
    ]

sc.pl.dotplot(temp,
              gene_vec,
              groupby = 'cell_type',
              standard_scale = 'var',
              categories_order = celltype_order,
              swap_axes = True)

In [ ]:
# UMAP colored by sample ID
sample_adata = adata.copy()

unique_samples = sample_adata.obs['orig.ident'].astype(str).unique()

colors = plt.cm.tab20(np.linspace(0, 1, len(unique_samples)))
sample_to_color = dict(zip(unique_samples, colors))

cell_colors = sample_adata.obs['orig.ident'].astype(str).map(sample_to_color)

x_subset = sample_adata.obsm['X_umap'][:, 0]
y_subset = sample_adata.obsm['X_umap'][:, 1]

plt.figure(figsize=(12, 12))
plt.scatter(x_subset, y_subset, c=cell_colors, s=0.5)

plt.xlabel('UMAP1')
plt.ylabel('UMAP2')
plt.title('All samples')

legend_elements = [Line2D([0], [0], marker='o', color='w',
                           markerfacecolor=sample_to_color[s], markersize=8, label=s)
                   for s in unique_samples]
plt.legend(handles=legend_elements, bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(f'figures/{d}_UMAP_samples.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# UMAP colored by cell type with custom color palette
cmap = plt.get_cmap('Set3')

cell_type_to_color = {
    'Epithelial': cmap(5 / 12),
    'Ciliated': cmap(6 / 12),
    'Fibroblast': cmap(9 / 12),
    'SMC/Pericyte': cmap(8 / 12),
    'Endothelial': cmap(7 / 12),
    'Mast': cmap(2 / 12),
    'Macrophage': cmap(11 / 12),
    'T/NK': cmap(4 / 12),
}

def rgba_to_hex(rgba):
    return "#{:02x}{:02x}{:02x}".format(
        int(rgba[0] * 255),
        int(rgba[1] * 255),
        int(rgba[2] * 255)
    )

adata_copy = adata.copy()

cell_colors = adata_copy.obs['cell_type'].map(
    {k: rgba_to_hex(v) for k, v in cell_type_to_color.items()}
)

x_subset = adata_copy.obsm['X_umap'][:, 0]
y_subset = adata_copy.obsm['X_umap'][:, 1]

plt.figure(figsize=(10, 10))
plt.scatter(x_subset, y_subset, c=cell_colors, s=0.5)

plt.xlabel('UMAP1')
plt.ylabel('UMAP2')
plt.title('Cell types')

legend_elements = [Line2D([0], [0], marker='o', color='w',
                           markerfacecolor=rgba_to_hex(v), markersize=10, label=k)
                   for k, v in cell_type_to_color.items()]
plt.legend(handles=legend_elements, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f'figures/{d}_UMAP_celltypes.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Stacked bar chart of cell type proportions per sample
cross_tab = pd.crosstab(adata.obs['orig.ident'], adata.obs['cell_type'])

# Calculate per-sample proportions
pct_df = cross_tab.div(cross_tab.sum(axis=1), axis=0) * 100

desired_order = ['Epithelial',
                 'Ciliated',
                 'Fibroblast',
                 'SMC/Pericyte',
                 'Endothelial',
                 'Mast',
                 'Macrophage',
                 'T/NK',]

pct_df = pct_df.reindex(columns=desired_order)

cmap = plt.get_cmap('Set3')
colors_list = [cmap(i / 12) for i in [5, 6, 9, 8, 7, 2, 11, 4]]

ax = pct_df.plot(
    kind='bar',
    stacked=True,
    figsize=(10, 6),
    color=colors_list,
)
ax.set_xlabel('Sample')
ax.set_ylabel('Proportion (%)')
ax.set_title('Cell type composition per sample')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig(f'figures/{d}_barplot_celltype_composition.pdf', bbox_inches='tight')
plt.show()

pct_df.to_csv('RatioCellTypes-discovery.txt', sep='\t')

## 3. Epithelial Subtype Characterization

Load the epithelial subclustering results and generate figures characterizing the major epithelial subtypes: SOX9+ basal-like (S9), common cancer progenitor (L5_S9_DP), proliferating (KI67), and luminal populations.

In [ ]:
# Each section reloads adata independently from the source h5ad to avoid
# unintended side effects from in-place modifications in preceding sections.
adata = anndata.read_h5ad('./11_subclustering/epithelial-cluster.h5ad')

In [ ]:
adata.obs.shape

In [ ]:
pd.DataFrame(adata.obs['cell_type'].unique())

In [ ]:
temp = adata[adata.obs['cell_type'] != 'Doublet'].copy()

gene_vec = [
'MKI67',
'KRT8', 'KRT18', # EPITHELIAL
'ESR1', 'PAX8', 'FOXA2',
'SOX9', 'MMP7',
'LGR5', 'CDH2', 'DKK1', 'NOTUM', 'AXIN2', 'ALDH1A1', 'KLK11',
'KMO', 'EMID1', 'IHH', 'PHLDA1', 'SLC7A11', # SOX9 functionalis
'SLC26A7', 'LEFTY1', 'PTGS1', 'MSLN', 'WNT7A', # Luminal
'MUC12', 'CDC20B', 'CCNO', 'HES6', # pre-ciliated
'FOXJ1', 'PIFO', # CILIATED
'SCGB2A2', 'S100P', # glandular
'PAEP', 'CXCL14', 'DPP4', 'GPX3', # secretory
'VEGFA',
]

celltype_order = [
    'KI67',
    'S9',
    'L5_S9_DP',
    'Lumenal',
    '18T',
    '8H',
    '15T1',
    '15T2',
    '17T2',
    'Preciliated',
    'Ciliated',
    'Secretory',
]

celltype_order = [c for c in celltype_order if c in temp.obs['cell_type'].values]

sc.pl.dotplot(temp,
              gene_vec,
              groupby = 'cell_type',
              standard_scale = 'var',
              categories_order = celltype_order,
              swap_axes = True)

In [ ]:
# UMAP colored by epithelial subtype
cmap = plt.get_cmap('Paired')

sub_cell_type_to_color = {
    'S9': cmap(1 / 12),
    'L5_S9_DP': cmap(3 / 12),
    'KI67': cmap(7 / 12),
    'Lumenal': cmap(6 / 12),
    '17T2': cmap(2 / 12),
    '8H': cmap(2 / 12),
    '15T1': cmap(2 / 12),
    '15T2': cmap(2 / 12),
    '18T': cmap(2 / 12),
    'Ciliated': cmap(8 / 12),
    'Preciliated': cmap(9 / 12),
    'Secretory': cmap(0 / 12),
}


adata_copy = adata.copy()

In [ ]:
# Stacked bar chart of epithelial subtype proportions per sample
cross_tab = pd.crosstab(adata_copy.obs['orig.ident'], adata_copy.obs['cell_type'])

pct_df = cross_tab.div(cross_tab.sum(axis=1), axis=0) * 100

desired_order = ['S9',
                 'L5_S9_DP',
                 'KI67',
                 'Lumenal',
                 '17T2',
                 '8H',
                 '15T1',
                 '15T2',
                 '18T',
                 'Ciliated',
                 'Preciliated',
                 'Secretory']

desired_order = [c for c in desired_order if c in pct_df.columns]
pct_df = pct_df.reindex(columns=desired_order)

colors_list = [rgba_to_hex(sub_cell_type_to_color.get(c, (0.5, 0.5, 0.5, 1.0))) for c in desired_order]

ax = pct_df.plot(kind='bar', stacked=True, figsize=(10, 6), color=colors_list)
ax.set_xlabel('Sample')
ax.set_ylabel('Proportion (%)')
ax.set_title('Epithelial subtype composition per sample')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig(f'figures/{d}_barplot_epithelial_subtype.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Violin plots of key marker genes per epithelial subtype
adata_copy = adata.copy()
adata_copy = adata_copy[adata_copy.obs['cell_type'] != 'Doublet']

adata_copy = adata_copy[~adata_copy.obs['cell_type'].isin(['Ciliated',
                                    'Preciliated',
                                    'Secretory'])]
adata_copy = adata_copy[~adata_copy.obs['patient'].isin([
                                    'AEH0005',
                                    'AEH0010'])]

In [ ]:
order = ['AEH0014_T1',
         'AEH0014_T2',
         'AEH0014_T3',
         'AEH0014_T4',
         'AEH0015_T1',
         'AEH0018_H3',
         'AEH0018_T',
         'AEH0007_T',
         'AEH0008_H',
         'AEH0015_T2',
         'AEH0017_T2',
         'AEH0019_T1',
         'AEH0019_T2',
         'AEH0019_T3',]

# Violin plots of LGR5, CDH2, SOX9 across all epithelial subtypes
fig, axes = plt.subplots(3, 1, figsize=(4, 4))

sc.pl.violin(adata_copy,
             keys='LGR5',
             groupby='orig.ident',
             use_raw=False,
             rotation=90,
             stripplot=True,
             jitter=True,
             size=1,
             ax=axes[0],
             palette=['#d3d3d3'],
             order = order,
             show=False)

sc.pl.violin(adata_copy,
             keys='CDH2',
             groupby='orig.ident',
             use_raw=False,
             rotation=90,
             stripplot=True,
             jitter=True,
             size=1,
             ax=axes[1],
             palette=['#d3d3d3'],
             order = order,
             show=False)

sc.pl.violin(adata_copy,
             keys='SOX9',
             groupby='orig.ident',
             use_raw=False,
             rotation=90,
             stripplot=True,
             jitter=True,
             size=1,
             ax=axes[2],
             palette=['#d3d3d3'],
             order = order,
             show=False)

plt.tight_layout()
plt.savefig(f'figures/{d}_violin_LGR5_CDH2_SOX9.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Violin plots in S9 cells: NOTUM, RUNX2
fig, axes = plt.subplots(3, 1, figsize=(4, 4))

sc.pl.violin(adata_copy[adata_copy.obs['cell_type']=='S9'],
             keys='NOTUM',
             groupby='orig.ident',
             use_raw=False,
             rotation=90,
             stripplot=True,
             jitter=True,
             size=1,
             ax=axes[0],
             palette=['#d3d3d3'],
             order = order,
             show=False)

sc.pl.violin(adata_copy[adata_copy.obs['cell_type']=='S9'],
             keys='RUNX2',
             groupby='orig.ident',
             use_raw=False,
             rotation=90,
             stripplot=True,
             jitter=True,
             size=1,
             ax=axes[1],
             palette=['#d3d3d3'],
             order = order,
             show=False)

plt.tight_layout()
plt.savefig(f'figures/{d}_violin_S9_NOTUM_RUNX2.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Violin plots in KI67 cells
fig, axes = plt.subplots(3, 1, figsize=(4, 4))

sc.pl.violin(adata_copy[adata_copy.obs['cell_type']=='KI67'],
             keys='LGR5',
             groupby='orig.ident',
             use_raw=False,
             rotation=90,
             stripplot=True,
             jitter=True,
             size=1,
             ax=axes[0],
             palette=['#d3d3d3'],
             order = order,
             show=False)

sc.pl.violin(adata_copy[adata_copy.obs['cell_type']=='KI67'],
             keys='CDH2',
             groupby='orig.ident',
             use_raw=False,
             rotation=90,
             stripplot=True,
             jitter=True,
             size=1,
             ax=axes[1],
             palette=['#d3d3d3'],
             order = order,
             show=False)

plt.tight_layout()
plt.savefig(f'figures/{d}_violin_KI67_LGR5_CDH2.pdf', bbox_inches='tight')
plt.show()

## 4. SOX9+ Basal-like Gene Signature — Discovery Cohort

Define the S9 gene signature by differential expression analysis between S9 and L5_S9_DP cells in the discovery cohort (group1 samples), and validate the signature by gene score computation across all epithelial cells.

In [ ]:
# Each section reloads adata independently from the source h5ad to avoid
# unintended side effects from in-place modifications in preceding sections.
adata = anndata.read_h5ad('./11_subclustering/epithelial-cluster.h5ad')
print(adata)
print('cell_type categories:', adata.obs['cell_type'].cat.categories.tolist())

In [ ]:
# Parameters
GROUP1_SAMPLES = ['AEH0014_T1', 'AEH0014_T2', 'AEH0014_T3',
                  'AEH0014_T4', 'AEH0015_T1', 'AEH0018_H3', 'AEH0018_T']

# Number of top DEGs to use as signature
N_GENES_FOR_SCORE = 30

# Filter thresholds
FILTER_NONZERO_FRAC = 0.1
FILTER_MEAN_EXPR    = 0.1

### Step 1: DEG Analysis (S9 vs L5_S9_DP, group1 only)

In [ ]:
# Restrict to group1 samples
adata_sub = adata[adata.obs['orig.ident'].isin(GROUP1_SAMPLES)].copy()

# Restrict to S9 / L5_S9_DP cells
adata_sub = adata_sub[adata_sub.obs['cell_type'].isin(['L5_S9_DP', 'S9'])].copy()
print(f'S9: {(adata_sub.obs.cell_type=="S9").sum()}, L5_S9_DP: {(adata_sub.obs.cell_type=="L5_S9_DP").sum()}')

# Gene filtering: retain genes with non-zero detection rate >= 10% and mean expression >= 0.1
nonzero_frac   = np.asarray((adata_sub.X > 0).sum(axis=0)).flatten() / adata_sub.n_obs
mean_expression = np.asarray(adata_sub.X.mean(axis=0)).flatten()

filter_mask = (nonzero_frac >= FILTER_NONZERO_FRAC) & (mean_expression >= FILTER_MEAN_EXPR)

print(f'Genes before filter: {adata_sub.n_vars}, after: {filter_mask.sum()}')
adata_temp = adata_sub[:, filter_mask].copy()

# DEG analysis
sc.tl.rank_genes_groups(adata_temp, groupby='cell_type', method='wilcoxon', reference='L5_S9_DP')
deg_results = sc.get.rank_genes_groups_df(adata_temp, group='S9')

# Volcano plot
deg_results['neg_log10_pval'] = -np.log10(deg_results['pvals_adj'] + 1e-300)

top_up   = deg_results.sort_values('logfoldchanges', ascending=False).head(5)
top_down = deg_results.sort_values('logfoldchanges').head(5)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(deg_results['logfoldchanges'], deg_results['neg_log10_pval'], color='grey', alpha=0.5, s=10)
ax.scatter(top_up['logfoldchanges'],   top_up['neg_log10_pval'],   color='red',  label='S9 up',      s=20)
ax.scatter(top_down['logfoldchanges'], top_down['neg_log10_pval'], color='blue', label='L5_S9_DP up', s=20)

for _, row in top_up.iterrows():
    ax.text(row['logfoldchanges'], row['neg_log10_pval'], row['names'], fontsize=8, color='red')
for _, row in top_down.iterrows():
    ax.text(row['logfoldchanges'], row['neg_log10_pval'], row['names'], fontsize=8, color='blue')

ax.set_xlabel('Log2 Fold Change (S9 vs L5_S9_DP)')
ax.set_ylabel('-Log10 Adjusted P-value')
ax.set_title('Volcano Plot: S9 vs L5_S9_DP (group1 only)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'figures/{d}_DEG_volcano_S9vsDP.pdf', bbox_inches='tight')
plt.show()
print('Saved volcano plot')

In [ ]:
# Display top DEGs
print('=== S9 upregulated (top 30) ===')
display(deg_results.sort_values('logfoldchanges', ascending=False).head(30))
print()
print('=== L5_S9_DP upregulated (top 30) ===')
display(deg_results.sort_values('logfoldchanges').head(30))

### Step 2: Signature Gene Definition

In [ ]:
# Extract top N DEGs as signature genes (padj < 0.05 only)
sig_deg = deg_results[deg_results['pvals_adj'] < 0.05].copy()

S9_GENES_DEG = (
    sig_deg.sort_values('logfoldchanges', ascending=False)
    .head(N_GENES_FOR_SCORE)['names'].tolist()
)
L5_S9_DP_GENES_DEG = (
    sig_deg.sort_values('logfoldchanges')
    .head(N_GENES_FOR_SCORE)['names'].tolist()
)

print(f'N_GENES_FOR_SCORE = {N_GENES_FOR_SCORE}')
print(f'\nS9_GENES_DEG ({len(S9_GENES_DEG)} genes):')
print(S9_GENES_DEG)
print(f'\nL5_S9_DP_GENES_DEG ({len(L5_S9_DP_GENES_DEG)} genes):')
print(L5_S9_DP_GENES_DEG)

### Step 3: Gene Signature Score Computation (group1 epithelial cells)

In [ ]:
# sc.tl.score_genes uses random reference gene sampling; scores are quantitatively
# robust across runs — the S9 vs L5_S9_DP discrimination is stable regardless of seed.
# Apply signature scores to all group1 epithelial cells
adata_g1 = adata[adata.obs['orig.ident'].isin(GROUP1_SAMPLES)].copy()
print(f'group1 cells: {adata_g1.n_obs}')
print(f'cell_type distribution:')
print(adata_g1.obs['cell_type'].value_counts())

# Compute signature scores
sc.tl.score_genes(adata_g1, gene_list=S9_GENES_DEG,      score_name='S9_score')
sc.tl.score_genes(adata_g1, gene_list=L5_S9_DP_GENES_DEG, score_name='L5_S9_DP_score')

# Summary statistics
for col in ['S9_score', 'L5_S9_DP_score']:
    print(f'\n--- {col} ---')
    print(adata_g1.obs.groupby('cell_type')[col].describe().round(3))

### Step 4: Visualization

In [ ]:
# UMAP: cell_type and signature scores
sc.pl.umap(adata_g1,
           color=['cell_type', 'S9_score', 'L5_S9_DP_score'],
           ncols=4,
           wspace=0.3,
           show=False)
plt.savefig(f'figures/{d}_UMAP_S9score_group1.pdf', bbox_inches='tight')
plt.show()
print('Saved UMAP')

In [ ]:
# Cell type ordering for plots
all_ct = adata_g1.obs['cell_type'].cat.categories.tolist()
priority = ['S9', 'L5_S9_DP']
ct_order = priority + sorted([c for c in all_ct if c not in priority])
ct_order = [c for c in ct_order if c in all_ct]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, score_col in zip(axes,
                          ['S9_score', 'L5_S9_DP_score']):
    sc.pl.violin(adata_g1, keys=score_col, groupby='cell_type',
                 order=ct_order, rotation=90, ax=ax, show=False)
    ax.set_title(score_col)
plt.tight_layout()
plt.savefig(f'figures/{d}_violin_S9score_by_celltype.pdf', bbox_inches='tight')
plt.show()
print('Saved violin plot')

In [ ]:
# Scatter: S9_score vs L5_S9_DP_score colored by cell type
fig, ax = plt.subplots(figsize=(6, 5))

palette_ct = {'S9': '#e41a1c', 'L5_S9_DP': '#377eb8'}
for ct in ct_order:
    mask = adata_g1.obs['cell_type'] == ct
    ax.scatter(
        adata_g1.obs.loc[mask, 'S9_score'],
        adata_g1.obs.loc[mask, 'L5_S9_DP_score'],
        label=ct, alpha=0.5, s=5,
        color=palette_ct.get(ct, '#aaaaaa')
    )

ax.axhline(0, color='grey', lw=0.5, ls='--')
ax.axvline(0, color='grey', lw=0.5, ls='--')
ax.set_xlabel('S9_score')
ax.set_ylabel('L5_S9_DP_score')
ax.set_title('S9 vs L5_S9_DP score (group1 cells)')
ax.legend(markerscale=3, bbox_to_anchor=(1, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(f'figures/{d}_scatter_S9vsDP_score.pdf', bbox_inches='tight')
plt.show()
print('Saved scatter plot')

In [ ]:
# Mann-Whitney U test: S9 vs L5_S9_DP scores
s9_cells  = adata_g1.obs[adata_g1.obs['cell_type'] == 'S9']
dp_cells  = adata_g1.obs[adata_g1.obs['cell_type'] == 'L5_S9_DP']

for score_col in ['S9_score', 'L5_S9_DP_score']:
    u_stat, p_val = mannwhitneyu(s9_cells[score_col], dp_cells[score_col], alternative='two-sided')
    print(f'{score_col}: S9 median={s9_cells[score_col].median():.3f}, '
          f'L5_S9_DP median={dp_cells[score_col].median():.3f}, '
          f'p={p_val:.2e}')

In [ ]:
# Fraction of cells with score > 0 and directional concordance
print('=== Fraction of cells with score > 0 ===')
for ct in ['S9', 'L5_S9_DP']:
    mask = adata_g1.obs['cell_type'] == ct
    for score_col in ['S9_score', 'L5_S9_DP_score']:
        pct = (adata_g1.obs.loc[mask, score_col] > 0).mean() * 100
        print(f'  {ct} / {score_col}: {pct:.1f}%')

# Directional concordance: S9 cells should have S9_score > L5_S9_DP_score and vice versa
for ct, expected_higher, expected_lower in [
        ('S9',      'S9_score', 'L5_S9_DP_score'),
        ('L5_S9_DP','L5_S9_DP_score','S9_score')]:
    mask = adata_g1.obs['cell_type'] == ct
    concordant = (adata_g1.obs.loc[mask, expected_higher] >
                  adata_g1.obs.loc[mask, expected_lower]).mean() * 100
    print(f'{ct}: {expected_higher} > {expected_lower} -> {concordant:.1f}% concordant')

In [ ]:
# Score distribution histogram
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, score_col in zip(axes, ['S9_score', 'L5_S9_DP_score']):
    for ct, color in [('S9','red'), ('L5_S9_DP','blue')]:
        vals = adata_g1.obs.loc[adata_g1.obs['cell_type']==ct, score_col]
        ax.hist(vals, bins=50, alpha=0.6, label=ct, color=color, density=True)
    ax.axvline(0, color='k', lw=1, ls='--')
    ax.set_xlabel(score_col)
    ax.set_ylabel('Density')
    ax.set_title(score_col)
    ax.legend()
plt.tight_layout()
plt.savefig(f'figures/{d}_histogram_S9score.pdf', bbox_inches='tight')
plt.show()
print('Saved histogram')

In [ ]:
# Dotplot of signature genes in S9 and L5_S9_DP cells
adata_focus = adata_g1[adata_g1.obs['cell_type'].isin(['S9', 'L5_S9_DP'])].copy()

all_sig_genes = (
    [g for g in S9_GENES_DEG if g in adata_focus.var_names] +
    [g for g in L5_S9_DP_GENES_DEG if g in adata_focus.var_names]
)
all_sig_genes = list(dict.fromkeys(all_sig_genes))  # Remove duplicates

sc.pl.dotplot(adata_focus, var_names=all_sig_genes, groupby='cell_type',
              standard_scale='var', swap_axes=True,
              show=False, figsize=(5, max(4, len(all_sig_genes)*0.25)))
plt.savefig(f'figures/{d}_dotplot_signature_genes.pdf', bbox_inches='tight')
plt.show()
print('Saved dotplot')

## 5. SOX9+ Basal-like Gene Signature — Validation Cohort

Apply the SOX9+ basal-like gene signature derived from the discovery cohort to the independent validation cohort epithelial cells, and quantitatively evaluate cross-cohort reproducibility using AUROC and rank-biserial correlation.

### Step 1: Derive Signature from Initial Cohort DEG

In [ ]:
# Load initial cohort epithelial cells
adata_init = anndata.read_h5ad('../discovery_cohort/11_subclustering/epithelial-cluster.h5ad')
print(f'Initial cohort: {adata_init.n_obs} cells x {adata_init.n_vars} genes')
print('cell_type:', adata_init.obs['cell_type'].value_counts().to_dict())

In [ ]:
# Restrict to group1 S9 / L5_S9_DP cells
adata_sub = adata_init[adata_init.obs['orig.ident'].isin(GROUP1_SAMPLES)].copy()
adata_sub = adata_sub[adata_sub.obs['cell_type'].isin(['L5_S9_DP', 'S9'])].copy()
print(f'S9: {(adata_sub.obs.cell_type=="S9").sum()}, L5_S9_DP: {(adata_sub.obs.cell_type=="L5_S9_DP").sum()}')

# Gene filtering
nonzero_frac    = np.asarray((adata_sub.X > 0).sum(axis=0)).flatten() / adata_sub.n_obs
mean_expression = np.asarray(adata_sub.X.mean(axis=0)).flatten()

filter_mask = (nonzero_frac >= FILTER_NONZERO_FRAC) & (mean_expression >= FILTER_MEAN_EXPR)

print(f'Genes before filter: {adata_sub.n_vars}, after: {filter_mask.sum()}')
adata_temp = adata_sub[:, filter_mask].copy()

# DEG analysis
sc.tl.rank_genes_groups(adata_temp, groupby='cell_type', method='wilcoxon', reference='L5_S9_DP')
deg_results = sc.get.rank_genes_groups_df(adata_temp, group='S9')
deg_results['neg_log10_pval'] = -np.log10(deg_results['pvals_adj'] + 1e-300)
print('DEG analysis complete.')

In [ ]:
# Extract top N signature genes (padj < 0.05)
sig_deg = deg_results[deg_results['pvals_adj'] < 0.05].copy()

S9_GENES_DEG = (
    sig_deg.sort_values('logfoldchanges', ascending=False)
    .head(N_GENES_FOR_SCORE)['names'].tolist()
)
L5_S9_DP_GENES_DEG = (
    sig_deg.sort_values('logfoldchanges')
    .head(N_GENES_FOR_SCORE)['names'].tolist()
)

print(f'N_GENES_FOR_SCORE = {N_GENES_FOR_SCORE}')
print(f'\nS9_GENES_DEG ({len(S9_GENES_DEG)} genes):')
print(S9_GENES_DEG)
print(f'\nL5_S9_DP_GENES_DEG ({len(L5_S9_DP_GENES_DEG)} genes):')
print(L5_S9_DP_GENES_DEG)

In [ ]:
# Volcano plot (discovery cohort reference)
top_up   = deg_results.sort_values('logfoldchanges', ascending=False).head(5)
top_down = deg_results.sort_values('logfoldchanges').head(5)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(deg_results['logfoldchanges'], deg_results['neg_log10_pval'], color='grey', alpha=0.5, s=10)
ax.scatter(top_up['logfoldchanges'],   top_up['neg_log10_pval'],   color='red',  label='S9 up',       s=20)
ax.scatter(top_down['logfoldchanges'], top_down['neg_log10_pval'], color='blue', label='L5_S9_DP up',  s=20)
for _, row in top_up.iterrows():
    ax.text(row['logfoldchanges'], row['neg_log10_pval'], row['names'], fontsize=8, color='red')
for _, row in top_down.iterrows():
    ax.text(row['logfoldchanges'], row['neg_log10_pval'], row['names'], fontsize=8, color='blue')
ax.set_xlabel('Log2 Fold Change (S9 vs L5_S9_DP)')
ax.set_ylabel('-Log10 Adjusted P-value')
ax.set_title('Volcano: S9 vs L5_S9_DP (Initial cohort group1)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'figures/{d}_DEG_volcano_initial.pdf', bbox_inches='tight')
plt.show()

### Step 2: Apply Signature to Validation Cohort

In [ ]:
# Each section reloads adata independently from the source h5ad to avoid
# unintended side effects from in-place modifications in preceding sections.
# Load validation cohort epithelial cells
adata = anndata.read_h5ad('./11_subclustering/epithelial-cluster.h5ad')
print(adata)
print('\ncell_type distribution:')
print(adata.obs['cell_type'].value_counts())
print('\norig.ident:', adata.obs['orig.ident'].unique().tolist())

In [ ]:
# sc.tl.score_genes uses random reference gene sampling; scores are quantitatively
# robust across runs — the S9 vs L5_S9_DP discrimination is stable regardless of seed.
# Compute signature scores for all validation cohort epithelial cells
sc.tl.score_genes(adata, gene_list=S9_GENES_DEG,       score_name='S9_score')
sc.tl.score_genes(adata, gene_list=L5_S9_DP_GENES_DEG, score_name='L5_S9_DP_score')

# Summary statistics
for col in ['S9_score', 'L5_S9_DP_score']:
    print(f'\n--- {col} ---')
    print(adata.obs.groupby('cell_type')[col].describe().round(3))

### Step 3: Visualization

In [ ]:
# Cell type display order
all_ct = adata.obs['cell_type'].cat.categories.tolist()
priority = ['S9', 'L5_S9_DP']
ct_order = priority + sorted([c for c in all_ct if c not in priority])
ct_order = [c for c in ct_order if c in all_ct]
print('ct_order:', ct_order)

In [ ]:
# UMAP: cell_type and signature scores
sc.pl.umap(adata,
           color=['cell_type', 'S9_score', 'L5_S9_DP_score'],
           ncols=4,
           wspace=0.3,
           show=False)
plt.savefig(f'figures/{d}_UMAP_S9score_validation.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Violin plots by cell type
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, score_col in zip(axes, ['S9_score', 'L5_S9_DP_score']):
    sc.pl.violin(adata, keys=score_col, groupby='cell_type',
                 order=ct_order, rotation=90, ax=ax, show=False)
    ax.set_title(score_col)
plt.tight_layout()
plt.savefig(f'figures/{d}_violin_S9score_by_celltype_validation.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Scatter: S9_score vs L5_S9_DP_score
palette_ct = {'S9': '#e41a1c', 'L5_S9_DP': '#377eb8'}

fig, ax = plt.subplots(figsize=(6, 5))
for ct in ct_order:
    mask = adata.obs['cell_type'] == ct
    ax.scatter(
        adata.obs.loc[mask, 'S9_score'],
        adata.obs.loc[mask, 'L5_S9_DP_score'],
        label=ct, alpha=0.4, s=5,
        color=palette_ct.get(ct, '#aaaaaa')
    )
ax.axhline(0, color='grey', lw=0.5, ls='--')
ax.axvline(0, color='grey', lw=0.5, ls='--')
ax.set_xlabel('S9_score')
ax.set_ylabel('L5_S9_DP_score')
ax.set_title('S9 vs L5_S9_DP score (Validation cohort)')
ax.legend(markerscale=3, bbox_to_anchor=(1, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(f'figures/{d}_scatter_S9vsDP_score_validation.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Mann-Whitney U test (Validation cohort)
s9_cells = adata.obs[adata.obs['cell_type'] == 'S9']
dp_cells = adata.obs[adata.obs['cell_type'] == 'L5_S9_DP']

print('=== Mann-Whitney (Validation cohort) ===')
for score_col in ['S9_score', 'L5_S9_DP_score']:
    u_stat, p_val = mannwhitneyu(s9_cells[score_col], dp_cells[score_col], alternative='two-sided')
    print(f'{score_col}: S9 median={s9_cells[score_col].median():.3f}, '
          f'L5_S9_DP median={dp_cells[score_col].median():.3f}, p={p_val:.2e}')

In [ ]:
# Directional concordance (Validation)
print('=== Fraction of cells with score > 0 (Validation) ===')
for ct in ['S9', 'L5_S9_DP']:
    mask = adata.obs['cell_type'] == ct
    for score_col in ['S9_score', 'L5_S9_DP_score']:
        pct = (adata.obs.loc[mask, score_col] > 0).mean() * 100
        print(f'  {ct} / {score_col}: {pct:.1f}%')

print()
print('=== Directional concordance (Validation) ===')
for ct, exp_high, exp_low in [
        ('S9',       'S9_score', 'L5_S9_DP_score'),
        ('L5_S9_DP', 'L5_S9_DP_score', 'S9_score')]:
    mask = adata.obs['cell_type'] == ct
    concordant = (adata.obs.loc[mask, exp_high] >
                  adata.obs.loc[mask, exp_low]).mean() * 100
    print(f'  {ct}: {exp_high} > {exp_low} -> {concordant:.1f}% concordant')

In [ ]:
# Score distribution histogram (Validation)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, score_col in zip(axes, ['S9_score', 'L5_S9_DP_score']):
    for ct, color in [('S9', 'red'), ('L5_S9_DP', 'blue')]:
        vals = adata.obs.loc[adata.obs['cell_type'] == ct, score_col]
        ax.hist(vals, bins=50, alpha=0.6, label=ct, color=color, density=True)
    ax.axvline(0, color='k', lw=1, ls='--')
    ax.set_xlabel(score_col)
    ax.set_ylabel('Density')
    ax.set_title(f'{score_col} (Validation)')
    ax.legend()
plt.suptitle('Score distribution: Validation cohort', y=1.02)
plt.tight_layout()
plt.savefig(f'figures/{d}_histogram_S9score_validation.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Dotplot of signature genes in S9 and L5_S9_DP cells (Validation)
adata_focus = adata[adata.obs['cell_type'].isin(['S9', 'L5_S9_DP'])].copy()

all_sig_genes = (
    [g for g in S9_GENES_DEG if g in adata_focus.var_names] +
    [g for g in L5_S9_DP_GENES_DEG if g in adata_focus.var_names]
)
all_sig_genes = list(dict.fromkeys(all_sig_genes))

sc.pl.dotplot(adata_focus, var_names=all_sig_genes, groupby='cell_type',
              standard_scale='var', swap_axes=True,
              show=False, figsize=(5, max(4, len(all_sig_genes) * 0.25)))
plt.savefig(f'figures/{d}_dotplot_signature_genes_validation.pdf', bbox_inches='tight')
plt.show()

### Step 4: Quantitative Evaluation (AUROC / Effect Size / Classification Accuracy)

In [ ]:
def compute_metrics(adata_scored, label='cohort'):
    """Compute quantitative metrics for S9 score discriminative performance."""
    sub = adata_scored[adata_scored.obs['cell_type'].isin(['S9', 'L5_S9_DP'])].copy()
    y_true  = (sub.obs['cell_type'] == 'S9').astype(int)
    s9_sc   = sub.obs['S9_score'].values
    dp_sc   = sub.obs['L5_S9_DP_score'].values
    n_s9    = (y_true == 1).sum()
    n_dp    = (y_true == 0).sum()

    auroc_s9 = roc_auc_score(y_true,  s9_sc)
    auroc_dp = roc_auc_score(y_true, -dp_sc)

    u_s9, _ = mannwhitneyu(s9_sc[y_true==1], s9_sc[y_true==0], alternative='two-sided')
    u_dp, _ = mannwhitneyu(dp_sc[y_true==0], dp_sc[y_true==1], alternative='two-sided')
    r_s9 = 2 * u_s9 / (n_s9 * n_dp) - 1   # >0: S9 cells score higher
    r_dp = 2 * u_dp / (n_dp * n_s9) - 1   # >0: DP cells score higher

    y_pred  = (s9_sc > dp_sc).astype(int)
    f1      = f1_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)

    return {
        'label': label,
        'n_S9': int(n_s9), 'n_DP': int(n_dp),
        'AUROC_S9': auroc_s9, 'AUROC_DP': auroc_dp,
        'r_S9': r_s9, 'r_DP': r_dp,
        'F1': f1, 'BalAcc': bal_acc,
        'y_true': y_true, 's9_sc': s9_sc, 'dp_sc': dp_sc,
    }

# Apply scores to initial cohort cells (training set reference)
sc.tl.score_genes(adata_sub, gene_list=S9_GENES_DEG,       score_name='S9_score')
sc.tl.score_genes(adata_sub, gene_list=L5_S9_DP_GENES_DEG, score_name='L5_S9_DP_score')

m_init = compute_metrics(adata_sub, label='Initial (group1, train)')
m_val  = compute_metrics(adata,     label='Validation')

# Summary table
metrics_df = pd.DataFrame([
    {'Metric': 'N (S9)',                    m_init['label']: m_init['n_S9'],            m_val['label']: m_val['n_S9']},
    {'Metric': 'N (L5_S9_DP)',              m_init['label']: m_init['n_DP'],            m_val['label']: m_val['n_DP']},
    {'Metric': 'AUROC (S9_score)',          m_init['label']: round(m_init['AUROC_S9'],3), m_val['label']: round(m_val['AUROC_S9'],3)},
    {'Metric': 'AUROC (L5_S9_DP_score)',   m_init['label']: round(m_init['AUROC_DP'],3), m_val['label']: round(m_val['AUROC_DP'],3)},
    {'Metric': 'Rank-biserial r (S9_score)',  m_init['label']: round(m_init['r_S9'],3),   m_val['label']: round(m_val['r_S9'],3)},
    {'Metric': 'Rank-biserial r (DP_score)',  m_init['label']: round(m_init['r_DP'],3),   m_val['label']: round(m_val['r_DP'],3)},
    {'Metric': 'F1 (S9_score > DP_score)', m_init['label']: round(m_init['F1'],3),  m_val['label']: round(m_val['F1'],3)},
    {'Metric': 'Balanced Accuracy',         m_init['label']: round(m_init['BalAcc'],3),   m_val['label']: round(m_val['BalAcc'],3)},
]).set_index('Metric')

print('=== Quantitative Evaluation: Initial vs Validation ===')
print('(Initial is the DEG training set; Validation is an independent cohort)')
display(metrics_df)

In [ ]:
# ROC curves: Initial cohort vs Validation cohort
fig, axes = plt.subplots(2, 2, figsize=(10, 9))

for row_idx, (m, cohort_label) in enumerate([(m_init, 'Initial (group1)'), (m_val, 'Validation')]):
    y_true = m['y_true']
    for col_idx, (score_vals, score_label, color, auroc) in enumerate([
            (m['s9_sc'],  'S9_score',       'red',  m['AUROC_S9']),
            (-m['dp_sc'], 'L5_S9_DP_score', 'blue', m['AUROC_DP']),
    ]):
        ax = axes[row_idx][col_idx]
        fpr, tpr, _ = roc_curve(y_true, score_vals)
        ax.plot(fpr, tpr, color=color, lw=2, label=f'AUROC = {auroc:.3f}')
        ax.plot([0, 1], [0, 1], 'k--', lw=1)
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title(f'{cohort_label}\n{score_label} (S9 vs L5_S9_DP)')
        ax.legend(loc='lower right')
        ax.set_aspect('equal')

plt.tight_layout()
plt.savefig(f'figures/{d}_ROC_S9score_initial_vs_validation.pdf', bbox_inches='tight')
plt.show()

### Methods Note

**Gene Signature Scoring**

Differential gene expression (DGE) analysis was performed between S9 and L5_S9_DP cells restricted to the initial cohort (group1 samples: AEH0014_T1/T2/T3/T4, AEH0015_T1, AEH0018_H3/T). Genes were filtered to retain those with non-zero detection rate >= 10% and mean expression >= 0.1. DGE was performed using the Wilcoxon rank-sum test (scanpy `sc.tl.rank_genes_groups`), with L5_S9_DP cells as the reference group. The top 30 genes upregulated in S9 (S9_GENES) and top 30 genes upregulated in L5_S9_DP (L5_S9_DP_GENES) were selected from genes with adjusted p-value < 0.05.

Gene signature scores were computed using `sc.tl.score_genes` (scanpy), which calculates mean expression of signature genes relative to a randomly sampled reference gene set of equal size. Cross-cohort validation was performed by applying the discovery cohort-derived signature to the independent validation cohort.

**Quantitative Evaluation**

Discriminative performance was assessed using (i) AUROC, (ii) rank-biserial correlation (*r* = 2U/(n1*n2) - 1, derived from Mann-Whitney U), and (iii) balanced accuracy using the decision rule S9 score > L5_S9_DP score to predict S9 identity. Effect size thresholds: small (*r* = 0.1), medium (*r* = 0.3), large (*r* = 0.5).